# NB-27 — the last 20 utterances of the 100-hour final checkpoint

**The smallest outstanding item, and the easiest to close.** The converged
100-hour checkpoint was measured on 80 of the 100 attack utterances. Because
batches are duration-sorted, the 20 that are missing are the 20 **longest**, so
the partial number is not comparable with any full one. The paper therefore
excludes this checkpoint from every pooled fit and carries a footnote about it.

Twenty utterances closes it. After this run the footnote goes, the checkpoint
can enter the fits, and §5.1's "46% measured on 80 of its 100 utterances"
becomes an ordinary number.

**This is NB-21 with a one-line queue.** Same attack configuration, same eval
set, same targets. The resume path reads the existing 80 rows and measures only
what is missing.

**Attach the dataset carrying `eps50_rung100-final_T5_n100.csv`** — without it
this re-measures all 100 and wastes the run. The cell prints the resume source;
if it says `NO RESUME SOURCE`, stop and fix the attachment.

**Cost.** Twenty utterances, and they are the longest ones, so budget 1–5
GPU-hours rather than the 0.9 a naive pro-rata would suggest.


## 1 · Queue and the locked attack configuration

In [ ]:
NOTEBOOK = "NB-27-finish-100h-final"

# One condition. The tag must reproduce the existing filename exactly, or the
# resume finds nothing and all 100 utterances are measured again.
QUEUE = [dict(tag="rung100-final", ckpt_dataset="ckpt-xlsr-100h", rung=100,
              checkpoint="final")]

WALL_HOURS = 11.0
MIN_HOURS_TO_START = 0.5

EDIT_TYPE      = "T5"
N_EVAL         = 100
BATCH_SIZE     = 8
USE_AMP        = True
FIXED_EPS      = None
EPS_LO, EPS_HI = 1e-4, 2e-3
BISECT_ROUNDS  = 5
PROBE_STEPS    = 400
CONFIRM_STEPS  = 800
STALL_PATIENCE = 100
CHECK_EVERY    = 50
LR             = 1e-4
SEED           = 1337
VALIDATE_BATCHING = False

_LOCKED = dict(edit=EDIT_TYPE, n=N_EVAL, bs=BATCH_SIZE, amp=USE_AMP, fx=FIXED_EPS,
               lo=EPS_LO, hi=EPS_HI, br=BISECT_ROUNDS, ps=PROBE_STEPS,
               cs=CONFIRM_STEPS, sp=STALL_PATIENCE, ce=CHECK_EVERY, lr=LR, seed=SEED)
assert _LOCKED == dict(edit="T5", n=100, bs=8, amp=True, fx=None, lo=1e-4, hi=2e-3,
                       br=5, ps=400, cs=800, sp=100, ce=50, lr=1e-4, seed=1337), (
    "Attack settings were edited. The 20 new utterances must be measured exactly "
    "as the first 80 were, or the completed file mixes two configurations.")

print(f"{NOTEBOOK} | finishing {QUEUE[0]['tag']}")
print("  existing: 80 of 100 utterances, ASR 46.25% on that subset")
print("  the 20 missing are the LONGEST, because batches are duration-sorted,")
print("  which is why the partial number is not comparable to a full one.")

## 2 · Boilerplate

In [ ]:
import os, hashlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata, gc
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

# PROV holds only what is defined AT THIS POINT. In NB-05 the rung, edit type
# and bracket were module-level constants, so they could be read here; in a
# queue runner they are per-condition and are not bound yet. Reading them here
# is a NameError that fires three cells before anything expensive starts. The
# run-specific fields are added by the summary cell at the end.
# Project-wide Bengali normaliser. Defined HERE, in boilerplate, because the
# input-resolution cell asserts with it and in NB-05 it lived in the (later)
# target-construction cell -- which a queue runner does not have.
ZW = dict.fromkeys(map(ord, "\u200c\u200d"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

PROV = {"notebook": NOTEBOOK, "seed": SEED,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# Internet is off in this session, so `pip install transformers==4.44.2` cannot
# reach the index. Measured twice in this project's own logs (NB-12 and NB-13,
# 2026-09-30): the cell spends ~202 s timing out, prints two ERROR lines, fails,
# and leaves the preinstalled version in place anyway. Pure waste.
#
# So we do not attempt the pin. We import what is already present and RECORD the
# version instead -- the pin's purpose was reproducibility, and a recorded
# version serves that better than a failed install does.
#
# If internet is ever turned ON, restore the pin: 4.44.2 is the version every
# figure measured before 2026-09-30 was produced on, and mixing versions across
# figures is exactly what the provenance record exists to catch.
#
# `jiwer` is deliberately NOT imported: it is not preinstalled on Kaggle, and
# the only place this notebook would use it is a FIXED_EPS clean-WER branch
# that cannot execute (FIXED_EPS is None here) and already guards itself.
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa, numpy
print(f"transformers {transformers.__version__} | numpy {numpy.__version__} "
      f"| librosa {librosa.__version__}")
if transformers.__version__ != "4.44.2":
    print(f"  NOTE: this is NOT the project pin (4.44.2). Record "
          f"{transformers.__version__} against every number this run produces.")
PROV["transformers"] = transformers.__version__
PROV["numpy"] = numpy.__version__


## 3 · Inputs, eval set and frozen targets

In [ ]:
# ---------------------------------------------------------------------------
# Kaggle does not use one mount layout. Depending on the kernel you get
#     /kaggle/input/<name>                (older kernels)
#     /kaggle/input/datasets/<name>       (your own datasets, current)
#     /kaggle/input/competitions/<name>   (competition data)
# A flat glob only sees the wrapper dirs -- NB-01 failed with exactly this,
# reporting: Visible inputs: ['competitions', 'datasets']
# ---------------------------------------------------------------------------
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, ("Attach bn-asr-splits -- test.csv is needed to resolve audio paths "
                "for the 100 target utterances.\n"
                f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
print("splits:", SPLITS)

# ---------------------------------------------------------------------------
# EVAL SET AND TARGETS: read, never regenerated.
#
# Task B's targets are published inside every eps50-*-n100 dataset as
# task_b_targets_n100.csv. Reading them makes this run byte-comparable by
# construction rather than by reproducing a generator. The hash is checked.
# ---------------------------------------------------------------------------
TGT_CSV = next((f"{p}/task_b_targets_n100.csv" for p in inputs
                if os.path.exists(f"{p}/task_b_targets_n100.csv")), None)
assert TGT_CSV, ("task_b_targets_n100.csv not found. Attach ONE Task B result "
                 "dataset, e.g. zoayriaabedin/eps50-rung1-final-t5-n100.\n"
                 f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
TGT_SHA = hashlib.sha256(open(TGT_CSV, "rb").read()).hexdigest()
print("targets:", TGT_CSV)
print("targets sha256:", TGT_SHA)
assert TGT_SHA == "8711dc56202b33f90e769c9735231b0fa8cf27fcbc20151f5466bcd9cc2b1256", (
    "These are NOT the targets the six Task B runs used. The comparison against "
    "run 1's 75% would be meaningless.")
tg = pd.read_csv(TGT_CSV)
assert len(tg) == 100 and tg.utt_id.nunique() == 100

_full = pd.read_csv(f"{SPLITS}/test.csv")
# source_tokens / target_tokens are carried through: the results writer uses
# both, and eps50_per_token divides by target_tokens. Dropping them here would
# KeyError at save time, after the whole sweep had run.
ev = tg[["utt_id", "target_text", "source_tokens", "target_tokens"]].merge(
        _full, on="utt_id", how="left")
assert ev["path"].notna().all(), "some target utterances are absent from test.csv"
assert (ev.text.map(norm_bn).values == tg.text.map(norm_bn).values).all(), \
    "transcript mismatch between the target file and test.csv"
EV_CSV = TGT_CSV
PROV["attack_eval_csv"] = os.path.basename(TGT_CSV)
PROV["targets_sha256"] = TGT_SHA
PROV["fixed_eps"] = FIXED_EPS
PROV["bisect_rounds"] = BISECT_ROUNDS
# The checkpoint dataset and model_dir are PER CONDITION here, so they are not
# recorded once at the top: every result row carries its own `model_dir`, and the
# queue summary records the tags.
for _c in ("path","text","target_text","source_tokens","target_tokens","duration"):
    assert _c in ev.columns, f"ev is missing {_c}, which a later cell needs"
print(f"eval set: {len(ev)} utterances, manifest order (original 25 first)")

# ---------------------------------------------------------------------------
# Re-resolve the audio root, same hazard as NB-01: NB-00 baked in the mount
# point from ITS session. Here the whole attack set is only 25 files, so check
# every one of them rather than sampling.
# ---------------------------------------------------------------------------
def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0):
        return df, os.path.dirname(p0)
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}",
                           f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, (f"Audio directory '{stem}' not found under /kaggle/input.\n"
                   "Attach: + Add Input -> Competitions -> 'bengaliai speech'.\n"
                   f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    df = df.copy()
    df["path"] = cands[0] + "/" + df["utt_id"].astype(str) + ext
    return df, cands[0]

ev, AUDIO_ROOT = resolve_audio(ev)
absent = [p for p in ev["path"] if not os.path.exists(p)]
assert not absent, f"{len(absent)}/{len(ev)} attack audio files missing, e.g.\n  {absent[0]}"
print(f"audio root: {AUDIO_ROOT}")
print(f"attack set: {len(ev)} utterances, all present")

## 4 · Front end, loss, PGD, bisection

Byte-identical to NB-05. These read `model`, `processor`, `BLANK_ID` and `DO_NORMALIZE` as globals at call time, which is what lets Stage 1 swap checkpoints without redefining anything.

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

In [ ]:
def pgd_batch(x, mask, tgt_pad, tgt_len, targets, eps_vec,
              steps, delta_init=None, lr=LR):
    """eps_vec: (B,) per-sample L-inf budgets.
       Returns (success bool array, frozen deltas, steps_used)."""
    B = x.shape[0]
    eps_col = eps_vec.view(-1, 1)
    delta = (torch.zeros_like(x) if delta_init is None
             else torch.max(torch.min(delta_init.clone(), eps_col), -eps_col))
    delta.requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x)
    success = np.zeros(B, dtype=bool)
    best_loss = torch.full((B,), float("inf"), device=DEVICE)
    stall = torch.zeros(B, dtype=torch.long, device=DEVICE)

    for step in range(steps):
        if not active.any():
            break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)

        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if delta.grad is not None:
            delta.grad[~active] = 0          # finished samples stop moving
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

        with torch.no_grad():                # stall detection
            improved = per < best_loss - 1e-3
            best_loss = torch.minimum(best_loss, per.detach())
            stall = torch.where(improved, torch.zeros_like(stall), stall + 1)
            give_up = active & (stall > STALL_PATIENCE)
            active = active & ~give_up

        if (step + 1) % CHECK_EVERY == 0:
            hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach()
                        success[i] = True
                        active[i] = False
    return success, frozen, step + 1

In [ ]:
def eps50_batch(x, mask, tgt_pad, tgt_len, targets, verbose=True):
    """Per-sample geometric bisection for the smallest budget that still works.

    ---------------------------------------------------------------------------
    BUG FIXED 2026-09-22 (found in the rung-1 T5 run, 40% censoring).

    The previous version ended with

        ok, best, _ = pgd_batch(..., hi, CONFIRM_STEPS, delta_init=best)

    which REASSIGNED `best` wholesale to pgd_batch's return value. That return
    is a `frozen` buffer holding zeros for any sample that did not succeed on
    that particular pass -- so one unlucky final pass wiped out a delta that had
    already been verified at a larger budget. Those samples were then written
    out as censored, with SNR 70-86 dB (the signature of delta == 0) even though
    the ceiling probe had already proved them attackable.

    In the rung-1 run that discarded ~6 of 25 real measurements. Worse, it does
    so at a RATE THAT VARIES BY MODEL, so comparing median eps50 across rungs
    would have been comparing differently-biased subsamples -- straight into
    RQ1's headline.

    Fix: keep a running `best_eps` / `best_delta` that is only ever updated on a
    VERIFIED success. Nothing overwrites a confirmed result. Censoring now means
    exactly one thing: the attack failed at every budget tried, ceiling included.

    Side benefit: eps50 becomes a guaranteed UPPER BOUND on the true minimum.
    Optimiser noise can only make it conservative, never wrong in an
    unpredictable direction -- which is the property you want in a metric you
    are about to compare across four models.
    ---------------------------------------------------------------------------
    """
    B = x.shape[0]
    lo = torch.full((B,), EPS_LO, device=DEVICE)
    hi = torch.full((B,), EPS_HI, device=DEVICE)
    best_eps   = torch.full((B,), float("nan"), device=DEVICE)   # verified only
    best_delta = torch.zeros_like(x)

    def record(okt, eps_vec, d):
        """Adopt a result only when it is both successful and better."""
        nonlocal best_eps, best_delta
        better = okt & (torch.isnan(best_eps) | (eps_vec < best_eps))
        best_eps   = torch.where(better, eps_vec, best_eps)
        best_delta = torch.where(better.view(-1, 1), d, best_delta)

    # Ceiling probe. Failure here is genuine right-censoring.
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, hi, CONFIRM_STEPS)
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, hi, d)
    feasible = ok.copy()
    if verbose:
        print(f"  ceiling probe eps={EPS_HI:.1e}: {feasible.sum()}/{B} feasible, "
              f"{(~feasible).sum()} censored")

    for rnd in range(BISECT_ROUNDS):
        mid = torch.sqrt(lo * hi)
        ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                             PROBE_STEPS, delta_init=best_delta)
        ok = ok & feasible
        okt = torch.tensor(ok, device=DEVICE)
        hi = torch.where(okt, mid, hi)
        lo = torch.where(okt, lo, mid)
        record(okt, mid, d)
        if verbose:
            print(f"  round {rnd+1}/{BISECT_ROUNDS}: {ok.sum()}/{B} succeeded "
                  f"| median eps {mid.median().item():.3e}")

    # Budget-consistency round: one more bisection at the FULL step budget, so
    # the reported boundary is measured at the budget eps50 is defined at rather
    # than at the cheaper probe budget. See RUNLOG for why this matters.
    #
    # Skipped in FIXED_EPS mode: there is no bracket to refine. The ceiling probe
    # already ran at CONFIRM_STEPS, so the single reported budget is measured at
    # the full budget and needs no second pass.
    if BISECT_ROUNDS == 0:
        eps       = best_eps.cpu().numpy()
        confirmed = ~np.isnan(eps)
        if verbose:
            print(f"  -> FIXED_EPS {EPS_HI:.1e}: {confirmed.sum()}/{B} succeeded, "
                  f"{(~confirmed).sum()} resisted")
        return eps, np.ones(B), confirmed, best_delta

    mid = torch.sqrt(lo * hi)
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                         CONFIRM_STEPS, delta_init=best_delta)
    ok = ok & feasible
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, mid, d)
    if verbose:
        print(f"  budget-consistency round @ {CONFIRM_STEPS} steps: {ok.sum()}/{B}")

    eps       = best_eps.cpu().numpy()
    confirmed = ~np.isnan(eps)                 # censored == never succeeded
    width     = (hi / lo).cpu().numpy()
    if verbose:
        print(f"  -> {confirmed.sum()}/{B} measured, {(~confirmed).sum()} censored")
    return eps, width, confirmed, best_delta


## 5 · Per-condition loader

In [ ]:
def load_condition(cfg):
    """Resolve and load one checkpoint, rebinding the globals the attack closures read.

    The closures in the cells above look up `model`, `processor`, `BLANK_ID` and
    `DO_NORMALIZE` at CALL time, not definition time, so rebinding them here is
    all that is needed to point the whole machine at a different checkpoint.
    Every one of the four must be rebound: leaving a stale BLANK_ID or
    DO_NORMALIZE behind would silently attack the new model through the old
    model's decoding assumptions.
    """
    global model, processor, BLANK_ID, DO_NORMALIZE, MODEL_DIR, RUNG, CHECKPOINT
    ds = cfg["ckpt_dataset"]
    cands = [p for p in inputs if re.search(rf"{re.escape(ds)}$", os.path.basename(p))]
    assert len(cands) == 1, (
        f"Need exactly one input matching '{ds}', found {len(cands)}: "
        f"{[os.path.basename(c) for c in cands]}")
    ck = cands[0]
    if cfg["checkpoint"] == "final":
        MODEL_DIR = ck if os.path.exists(f"{ck}/config.json") else f"{ck}/final"
    else:
        MODEL_DIR = f"{ck}/milestones/step-{cfg['checkpoint'].split('-')[1]}"
    assert os.path.exists(f"{MODEL_DIR}/config.json"), f"no config.json under {MODEL_DIR}"

    processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    model.config.apply_spec_augment = False
    DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
    BLANK_ID = model.config.pad_token_id
    RUNG, CHECKPOINT = cfg["rung"], cfg["checkpoint"]
    print(f"  loaded {MODEL_DIR}")
    print(f"  do_normalize {DO_NORMALIZE} | blank {BLANK_ID} | vocab {model.config.vocab_size}")
    return MODEL_DIR


def unload():
    global model, processor
    try:
        del model, processor
    except NameError:
        pass
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 6 · Per-condition sweep, unchanged from NB-21

In [ ]:
def run_condition(cfg, deadline):
    """Run one condition to completion or to the deadline.

    Returns (status, rows). status is "complete", "partial" or "skipped".
    Results are written after EVERY batch, so a kill at any point loses at most
    one batch.
    """
    tag = cfg["tag"]
    OUT = f"{WORK}/eps50_{tag}_{EDIT_TYPE}_n{N_EVAL}.csv"

    # ---------------------------------------------------------------------
    # Resume: this session's own output first, then ANY attached source
    # carrying the same filename, searched at EVERY mount depth.
    #
    # The original globbed only `/kaggle/input/*/F` and `/kaggle/input/*/*/F`.
    # On Kaggle's current layout a dataset file lives at
    #     /kaggle/input/datasets/<owner>/<name>/F
    # which is FOUR components deep, so neither pattern could ever match it --
    # the resume path was dead code, and a re-commit would silently re-measure
    # all 100 utterances instead of the 20 outstanding. Stage 2's own discovery
    # already searched to that depth; the resume did not, and no run had
    # exercised it, so nothing caught the mismatch.
    #
    # A notebook-output attachment may mount at a different depth again, so this
    # sweeps 1 to 4 and PRINTS every source it used. A silent zero is the failure
    # mode that matters here, so the no-source case says so loudly.
    # ---------------------------------------------------------------------
    results = []
    _base = os.path.basename(OUT)
    srcs = [OUT] if os.path.exists(OUT) else []
    for _d in range(1, 5):
        srcs += sorted(glob.glob("/kaggle/input/" + "*/" * _d + _base))
    srcs = list(dict.fromkeys(srcs))          # keep order, drop repeats
    seen = set()
    for p in srcs:
        _n = 0
        for r in pd.read_csv(p).to_dict("records"):
            if r["utt_id"] not in seen:
                seen.add(r["utt_id"]); results.append(r); _n += 1
        print(f"  resume source: +{_n:3d} new rows  {p}")
    if results:
        print(f"  resuming: {len(results)} utterances already measured")
    else:
        print(f"  NO RESUME SOURCE for {_base} -- starting from scratch.")
        print("    If you attached a previous partial run and expected to continue,")
        print("    stop and check it is visible: the previous session's Stage 2")
        print("    listing prints the exact filename this looks for.")
    if len(results) >= len(ev):
        print(f"  ALREADY COMPLETE ({len(results)}/{len(ev)}) -- skipping")
        pd.DataFrame(results).to_csv(OUT, index=False)
        return "skipped", results

    todo = ev[~ev.utt_id.isin(seen)].reset_index(drop=True)
    batches = [todo.iloc[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    print(f"  {len(todo)} remaining -> {len(batches)} batches of up to {BATCH_SIZE}")

    for bi, sub in enumerate(batches):
        if time.time() > deadline:
            print(f"  *** wall clock reached, {sum(len(b) for b in batches[bi:])} "
                  f"utterances unmeasured in this condition ***")
            return "partial", results
        xb, mb = load_batch(sub)
        tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE)
              for t in sub.target_text]
        tl = torch.tensor([len(t) for t in tp], device=DEVICE)
        tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
        t0 = time.time()
        eps, width, conf, best = eps50_batch(xb, mb, tpad, tl, list(sub.target_text),
                                             verbose=False)
        dt = time.time() - t0
        x_adv = int16_rt(torch.clamp(xb + best, -1, 1)) * mb
        snrs = snr_db_b(xb, x_adv, mb)
        hyp = decode_b(x_adv, mb)
        for i, (_, r) in enumerate(sub.iterrows()):
            ok = not np.isnan(eps[i])
            results.append(dict(
                utt_id=r["utt_id"], tag=tag, rung=RUNG, edit_type=EDIT_TYPE,
                checkpoint=CHECKPOINT, duration=r["duration"], source_text=r["text"],
                target_text=r["target_text"], source_tokens=int(r["source_tokens"]),
                target_tokens=int(r["target_tokens"]),
                eps50=float(eps[i]) if ok else None, censored=not ok,
                eps50_per_token=(float(eps[i]) / r["target_tokens"]) if ok else None,
                bracket_width=float(width[i]), snr_db=float(snrs[i]),
                decoded=(hyp[i] if ok else None),
                exact_match=(bool(hyp[i] == r["target_text"]) if ok else False),
                use_amp=USE_AMP, model_dir=MODEL_DIR))
        pd.DataFrame(results).to_csv(OUT, index=False)
        rem = len(batches) - (bi + 1)
        eta = time.time() + rem * dt
        print(f"    batch {bi+1}/{len(batches)}  {dt/60:4.1f} min  "
              f"{int((~np.isnan(eps)).sum())}/{len(sub)} uncensored  "
              f"{len(results)}/{len(ev)} rows"
              + (f"  | {rem} left, ~{rem*dt/3600:.1f} h"
                 f" ({'fits' if eta < deadline else 'WILL NOT FIT'})" if rem else ""))
    return ("complete" if len(results) >= len(ev) else "partial"), results

## 7 · Run

In [ ]:
SESSION_START = time.time()
DEADLINE = SESSION_START + WALL_HOURS * 3600
summary = []

for qi, cfg in enumerate(QUEUE, 1):
    left = (DEADLINE - time.time()) / 3600
    print("\n" + "=" * 74)
    print(f"[{qi}/{len(QUEUE)}] {cfg['tag']}   ({left:.1f} h of wall clock left)")
    print("=" * 74)
    if left < MIN_HOURS_TO_START:
        print(f"  only {left:.1f} h left, below MIN_HOURS_TO_START={MIN_HOURS_TO_START}.")
        print("  NOT STARTING. Publish /kaggle/working, attach it back, re-commit.")
        summary.append(dict(tag=cfg["tag"], status="not-started", n=0, asr=None))
        continue
    try:
        load_condition(cfg)
        status, rows = run_condition(cfg, DEADLINE)
    except Exception as e:
        print(f"  !! {type(e).__name__}: {e}")
        summary.append(dict(tag=cfg["tag"], status=f"error: {type(e).__name__}",
                            n=0, asr=None))
        unload(); continue
    df = pd.DataFrame(rows)
    asr = float((~df.censored.astype(bool)).mean() * 100) if len(df) else None
    summary.append(dict(tag=cfg["tag"], status=status, n=len(df), asr=asr))
    print(f"  {status.upper()}  {len(df)}/{len(ev)} measured"
          + (f"  ASR@2e-3 = {asr:.1f}%" if asr is not None else ""))
    unload()

print("\n" + "=" * 74)
print(f"SESSION DONE in {(time.time()-SESSION_START)/3600:.2f} h")
print("=" * 74)

## 8 · Result and what it unblocks

In [ ]:
sm = pd.DataFrame(summary)
print(sm.to_string(index=False))

for r in sm.itertuples():
    if r.asr is None:
        continue
    print(f"\n  {r.tag}: {r.n}/100 measured, ASR@2e-3 = {r.asr:.1f}%")
    if r.n >= 100:
        print(f"  Partial figure was 46.25% on the 80 SHORTEST utterances.")
        print(f"  Full figure is {r.asr:.1f}%. The difference is the duration bias.")
        print("\n  Now do these, in order:")
        print("   1. add `100h final` to COND in analysis/reanalysis.py")
        print("      (hours=98.02, trun='100h-r1', env='tf4.44.2', grp=None)")
        print("   2. rerun it: the fit gains a condition, G is unchanged at 12")
        print("      because this checkpoint belongs to a run already in the fit")
        print("   3. drop the 'excluded throughout because only 80 of its 100")
        print("      utterances were measured' note from tab:ancova's caption")
        print("   4. §5.1: '46% (measured on 80 of its 100 utterances)' becomes")
        print(f"      '{r.asr:.0f}%' with no caveat")
    else:
        print(f"  STILL PARTIAL. Publish, attach the output back, re-commit.")

PROV.update(notebook=NOTEBOOK, summary=summary,
            session_hours=round((time.time()-SESSION_START)/3600, 2))
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), ensure_ascii=False,
          indent=2, default=str)
json.dump({"title": "eps50-rung100-final-complete",
           "id": "YOURUSERNAME/eps50-rung100-final-complete",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
for f in sorted(os.listdir(WORK)):
    print("   ", f)
print("\nCHECK THIS FILE LIST before publishing: publishing REPLACES the dataset.")